<a href="https://colab.research.google.com/github/vyenn/ML2024/blob/main/docs/DNN-Lab-6-Batchnorm-and-Convnets-student-version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

This exercise covers two aspects:
* In tasks 1-6 you will implement mechanisms that allow training deeper models. After doing each of the tasks you can look at the plots and check how your changes impact gradients of network layers.
* In task 7 you will implement a convnet using [conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and try to reach 99% accuracy on MNIST.

For this notebook, running on CPU is sufficient: tasks 1–5. run under a minute, 6. under two minutes, and 7. up to 5 minutes.

Tasks:
1. Check that the given implementation reaches 95% test accuracy for
   the architecture `input-64-64-10` in 8 epochs.
2. Modify `Linear.reset_parameters()` to improve initialization and
   check that the network learns much faster and reaches over 97% test accuracy.
   A good basic initialization scheme is so-called *Glorot initialization*.
   For a linear layer with $n_{in}$ and  $n_{out}$ input and output features,
   it samples each weight from a normal distribution with $0$ mean and a standard deviation of $\sqrt{\frac{2}{n_{in}+n_{out}}}$.  
   Check how that changes the distribution of gradients at the first epoch.
3. Check, that with proper initialization we can train the architecture
   `input-64-64-64-64-64-64-64-10` (seven hidden layers of `64`), for e.g. 3 epochs,
   while with the original, bad initialization, it does not even get off the ground.
4. Implement Dropout as a PyTorch module (without using torch.nn.Dropout).
   Use `p=0.25` as the zeroing probability.
   Use Dropout after activations of each layer, except the last.
5. Check that with 10 hidden layers (64 units each), even with proper initialization,
   but with dropout added, the network has a hard time to start learning.
6. Implement batch normalization (without using torch.nn.BatchNorm).
    * compute batch mean and variance
    * add new variables beta and gamma
    * check that the network learns even for 10 hidden layers.
    * check how gradients change in comparison to network without batch norm.
   
   In pseudo-code:
    ```
		if training:
		    var, mean = torch.var_mean(x, dim=/batch/)
		    update running_mean, running_var
		else:
			mean, var = running_mean, running_var
		x = (x - mean) / sqrt(var + eps)
		return x * gamma + beta
    ```

7. Design and implement a simple convolutional network and achieve 99% test accuracy (in up to 10 epochs, 5 should be enough). The architecture is up to you, but even a few convolutional layers should be enough.
You may use `torch.nn.Conv2d, BatchNorm2d, MaxPool2d, ReLU, Dropout2d, Linear, Flatten, BatchNorm1d`.
Don't make it too generic, just try a few fixed layers.

In [3]:
import math
import sys

import numpy as np
import PIL.Image
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from plotly.subplots import make_subplots
from torch import Tensor
from torch.nn import init

if 'google.colab' in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

In [4]:
# @title Visualize gradients

class GradientVisualizer:
    def __init__(self, net: nn.Module, num_epochs: int) -> None:
        self.num_epochs = num_epochs
        self.linear_layers = self.get_linear_layers(net)

        self.grad_to_weight_fig = None
        self.grads_in_layers_fig = None
        self.grads_at_epochs_fig = None
        self.init_figures()

    @staticmethod
    def get_linear_layers(net):
        linear_layers = []
        for field in net.modules():
            if isinstance(field, nn.Linear | Linear):
                linear_layers.append(field)
        assert linear_layers, "No linear layers found. Forgot to use Sequential or ModuleList?"
        return linear_layers

    def get_epochs_for_one_layer(self):
        """
        We want to show gradient distributions from up to 7 selected epochs
        for one linear layer.
        """
        if self.num_epochs < 7:
            return list(range(self.num_epochs))
        else:
            return torch.linspace(0, self.num_epochs - 1, 7).int().tolist()

    def get_three_epochs(self):
        """
        We want to show gradients distributions from all layers at each of
        three epochs: first, middle and last.
        """
        return [0, self.num_epochs // 2, self.num_epochs - 1]

    def rgb_to_rgba(self, rgb_color, epoch):
        """
        Value of epoch parameter determines how transparent color should be
        in comparison to others.
        Colors for earlier epochs should be more transparent/less visible.
        """
        return f'rgba{rgb_color[3:-1]},{0.6 * (epoch + 1) / self.num_epochs + 0.15})'

    def init_figures(self):
        # Initialize figure with gradient to weight ratio plot
        fig = go.Figure()
        fig.update_layout(
            title='Gradient stddev to weight stddev ratio', title_x=0.5,
            xaxis_title='Epoch',
            yaxis_title='Gradient to weight ratio (log scale)',
            height=400, width=1500, margin=dict(b=10, t=60)
        )
        fig.update_yaxes(type='log')
        for i in range(len(self.linear_layers)):
            fig.add_trace(go.Scatter(
                x=[], y=[],
                mode='lines+markers', name=f'Linear layer {i}'
            ))

        self.grad_to_weight_fig = go.FigureWidget(fig)
        display(self.grad_to_weight_fig)

        # Initialize figure visualizing gradient distributions in layers
        chosen_layers = [0, len(self.linear_layers) // 2, len(self.linear_layers) - 1]
        num_rows = (len(chosen_layers) - 1) // 3 + 1
        fig = make_subplots(
            rows=num_rows, cols=3,
            subplot_titles=[f'Linear layer {i}' for i in chosen_layers],
            vertical_spacing=0.2 / num_rows
        )
        fig.update_layout(
            title='Gradient distributions by epoch (for three chosen layers)', title_x=0.5,
            height=num_rows * 400, width=1500, margin=dict(b=10, t=60)
        )

        colors, _ = px.colors.convert_colors_to_same_type(2 * px.colors.qualitative.Plotly)
        for i, layer_num in enumerate(chosen_layers):
            row = i // 3 + 1
            col = i % 3 + 1
            fig.update_xaxes(
                title_text='Gradient value', range=(-0.1, 0.1), row=row, col=col
            )
            fig.update_yaxes(
                title_text='Density (log scale)', type='log', row=row, col=col
            )

            # Create empty traces and update them later with actual gradient distributions.
            # Unfortunately, we cannot add new traces dynamically because Colab has problem
            # with widgets from plotly (traces added dynamically are rendered twice).
            for epoch in self.get_epochs_for_one_layer():
                fig.add_trace(
                    go.Scatter(
                        mode='lines', name=f'Epoch {epoch + 1}',
                        line=dict(color=self.rgb_to_rgba(colors[layer_num], epoch)),
                        legendgroup=layer_num
                    ),
                    row=row, col=col
                )

        self.grads_in_layers_fig = go.FigureWidget(fig)
        display(self.grads_in_layers_fig)

        # Initialize figure comparing gradient distributions between layers at the
        # first, middle and last epoch
        selected_epochs_indices = self.get_three_epochs()
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=[f'Epoch {epoch + 1}' for epoch in selected_epochs_indices]
        )
        fig.update_layout(
            title='Gradient distributions by layers (for three chosen epochs)', title_x=0.5,
            height=400, width=1500, margin=dict(b=10, t=60)
        )

        for col, epoch in enumerate(selected_epochs_indices, 1):
            fig.update_yaxes(title_text='Density (log scale)', type='log', row=1, col=col)
            fig.update_xaxes(
                title_text='Gradient value',
                range=(-0.05, 0.05) if epoch != 0 else (-1, 1),
                row=1, col=col
            )

            # Create empty traces and update them later with actual gradient distributions.
            for layer_num in range(len(self.linear_layers)):
                fig.append_trace(
                    go.Scatter(
                        mode='lines', name=f'Linear layer {layer_num}',
                        line=dict(color=colors[layer_num]), showlegend=(col == 1)
                    ),
                    row=1, col=col
                )

        self.grads_at_epochs_fig = go.FigureWidget(fig)
        display(self.grads_at_epochs_fig)

    def visualize_gradients(self, lr, epoch, batch_idx):
        # It is enough to use gradients calculated for the first batch.
        if batch_idx != 0:
            return

        epoch_grads = []
        epoch_grad_to_weight_ratios = []
        for layer in self.linear_layers:
            epoch_grads.append(layer.weight.grad.flatten().detach())
            epoch_grad_to_weight_ratios.append(
                (lr * layer.weight.grad.std() / layer.weight.std()).item()
            )

        # Update figure with gradient to weight ratio plot
        assert self.grad_to_weight_fig is not None
        for i, grad_to_weight_ratio in enumerate(epoch_grad_to_weight_ratios):
            x = self.grad_to_weight_fig.data[i].x
            next_x_val = x[-1] + 1 if x else 1
            self.grad_to_weight_fig.data[i].x += (next_x_val, )
            self.grad_to_weight_fig.data[i].y += (grad_to_weight_ratio, )

        # Update figure visualizing gradient distributions in layers
        assert self.grads_in_layers_fig is not None
        chosen_layers = [0, len(self.linear_layers) // 2, len(self.linear_layers) - 1]
        selected_epochs = self.get_epochs_for_one_layer()
        if epoch in selected_epochs:
            epoch_idx = selected_epochs.index(epoch)
            for layer_num, layer_grad in enumerate(epoch_grads):
                try:
                    layer_id = chosen_layers.index(layer_num)
                except ValueError:
                    continue
                trace_idx = layer_id * len(selected_epochs) + epoch_idx
                try:
                    hy, hx = torch.histogram(layer_grad, bins=50, density=True)
                except RuntimeError:
                    continue
                hy = hy / max(hy) + 0.001
                self.grads_in_layers_fig.data[trace_idx].x = hx[:-1].tolist()
                self.grads_in_layers_fig.data[trace_idx].y = hy.tolist()


        # Update figure visualizing gradient distributions at epochs
        assert self.grads_at_epochs_fig is not None
        selected_epochs = self.get_three_epochs()
        if epoch in selected_epochs:
            epoch_idx = selected_epochs.index(epoch)
            for layer_num, layer_grad in enumerate(epoch_grads):
                trace_idx = epoch_idx * len(self.linear_layers) + layer_num
                try:
                    hy, hx = torch.histogram(layer_grad, bins=50, density=True)
                except RuntimeError:
                    continue
                hy = hy / max(hy) + 0.001
                self.grads_at_epochs_fig.data[trace_idx].x = hx[:-1].tolist()
                self.grads_at_epochs_fig.data[trace_idx].y = hy.tolist()

In [5]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, glorot_init: bool) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(Tensor(out_features, in_features))
        self.bias = nn.Parameter(Tensor(out_features))
        self.glorot_init = glorot_init
        self.reset_parameters()

    def reset_parameters(self) -> None:
        if self.glorot_init:
            self.weight.data.normal_(mean=0, std=np.sqrt(2/(self.in_features + self.out_features)))
        else:
            self.weight.data.normal_(mean=0, std=0.25)
        init.zeros_(self.bias)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (*B, in_features)
        Output: shape (*B, out_features)
        """
        r = x.matmul(self.weight.T)
        r += self.bias
        return r


class Dropout(nn.Module):
    def __init__(self, p=0.25):
        """
        p is a probability of zeroing out a coefficient
        """
        assert 0 <= p < 1
        super().__init__()
        self.p = p

    def forward(self, x: Tensor) -> Tensor:
        if self.training:
            helper = torch.rand_like(x)
            mask = (helper > self.p).float()
            x = mask * x / (1-self.p)
        return x


class BatchNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x):
        if self.training:
            batch_mean = x.mean(dim=0)
            batch_var = x.var(dim=0, unbiased=False)

            # Update running estimates
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean  # detach batch_mean and batch_var if doesnt work
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var     # used to not work without it, but now it does...

            # Normalize
            x_hat = (x - batch_mean) / torch.sqrt(batch_var + self.eps)
        else:
            x_hat = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps)

        return x_hat * self.gamma + self.beta


class Net(nn.Module):
    def __init__(self, glorot_init: bool = False) -> None:
        super().__init__()

        self.fcs = nn.ModuleList([
            Linear(784, 64, glorot_init=glorot_init),
            *[Linear(64, 64, glorot_init=glorot_init) for _ in range(9)],
            Linear(64, 10, glorot_init=glorot_init)
        ])
        self.bns = nn.ModuleList([BatchNorm(64) for _ in range(10)])
        self.dropout = Dropout(0.25)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, H=28, W=28)
        Output: shape (B, num_classes=10)
        """
        x = x.view(-1, 28 * 28)
        for fc, bn in zip(self.fcs[:-1], self.bns):
            x = self.dropout(F.relu(bn(fc(x))))
        x = self.fcs[-1](x)
        return x

In [6]:
class MnistTrainer:
    def __init__(self, batch_size: int = 128) -> None:
        transform = transforms.Compose([transforms.ToTensor()])
        self.trainset = torchvision.datasets.MNIST(
            root='./data', download=True, train=True, transform=transform
        )
        self.trainloader = torch.utils.data.DataLoader(
            self.trainset, batch_size=batch_size, shuffle=True, num_workers=2
        )
        self.testset = torchvision.datasets.MNIST(
            root='./data', train=False, download=True, transform=transform
        )
        self.testloader = torch.utils.data.DataLoader(
            self.testset, batch_size=1000, shuffle=False, num_workers=2
        )

    def train(
        self,
        net: nn.Module,
        gradient_visualizer: GradientVisualizer | None = None,
        epochs: int = 20,
        lr: float = 0.05,
        momentum: float = 0.9
    ) -> None:
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)
        log_freq = 100

        for epoch in range(epochs):
            net.train()
            running_loss = 0.0
            for i, data in enumerate(self.trainloader):
                inputs, labels = data
                optimizer.zero_grad()

                outputs = net(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                if gradient_visualizer is not None:
                    gradient_visualizer.visualize_gradients(lr, epoch, i)
                optimizer.step()

                running_loss += loss.item()
                if i % log_freq == log_freq - 1:
                    running_loss /= log_freq
                    print(f"epoch {epoch + 1:>5}, batch {i + 1:>5}, loss {running_loss:.3f}")
                    running_loss = 0.0
            self.test(net)

    def test(self, net: nn.Module) -> list[float]:
        net.eval()
        correct = 0
        total = 0
        all_gt_logits = []
        with torch.no_grad():
            for data in self.testloader:
                images, labels = data
                outputs = net(images)
                gt_logits = torch.gather(outputs, dim=1, index=labels.unsqueeze(-1))
                all_gt_logits.extend(float(logit.item()) for logit in gt_logits)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Accuracy of the network on the {total} test images: {correct / total:.2%}")
        return all_gt_logits

    def show_worst_misclassified(self, net: nn.Module, k: int = 10) -> None:
        all_gt_logits = self.test(net)
        _, indices = torch.topk(-torch.tensor(all_gt_logits), k)
        imgs = []
        labels = []
        for i in indices:
            img, label = self.testset[int(i.item())]
            imgs.append(img)
            labels.append(label)
        image = PIL.Image.fromarray(
            torch.cat(imgs, dim=-1).mul(255).to(torch.uint8).permute(1, 2, 0).numpy().squeeze()
        )
        display(image.resize((5 * image.width, 5 * image.height), PIL.Image.Resampling.NEAREST))
        print(labels)

In [7]:
epochs = 8

net = Net(glorot_init=True)
gradient_visualizer = GradientVisualizer(net, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'fec66b9a-d898-4da2-944a-ae614f74ec48',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'd57e9f28-5c7b-4e6f-8ef9-0474bd97d9b5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': 'c550aebf-baf1-4661-b3bb-67b804975c4f',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '5de90c52-d1e0-43fa-af8a-ecff3813a347',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.22499999999999998)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '849a0531-5657-442a-9684-e4a6cbab3618',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.3)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': '9e7b114f-5cdf-4fba-8d09-1ad92efdd173',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.375)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'ab0e17b5-9f24-4b93-bf35-c763a59279a3',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
         

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '71a344f8-5a82-4380-807a-f5ec8e48bd46',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '78f3ce0f-35fc-4785-acee-4b4bf0ab8de5',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': '50722b6d-57af-4f5b-afdd-54e467954311',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

In [8]:
trainer = MnistTrainer(batch_size=128)
trainer.train(net, gradient_visualizer, epochs=epochs)

epoch     1, batch   100, loss 2.090
epoch     1, batch   200, loss 1.491
epoch     1, batch   300, loss 1.110
epoch     1, batch   400, loss 0.958
Accuracy of the network on the 10000 test images: 87.18%
epoch     2, batch   100, loss 0.751
epoch     2, batch   200, loss 0.707
epoch     2, batch   300, loss 0.666
epoch     2, batch   400, loss 0.624
Accuracy of the network on the 10000 test images: 92.63%
epoch     3, batch   100, loss 0.557
epoch     3, batch   200, loss 0.548
epoch     3, batch   300, loss 0.510
epoch     3, batch   400, loss 0.513
Accuracy of the network on the 10000 test images: 94.22%
epoch     4, batch   100, loss 0.480
epoch     4, batch   200, loss 0.481
epoch     4, batch   300, loss 0.478
epoch     4, batch   400, loss 0.454
Accuracy of the network on the 10000 test images: 94.72%
epoch     5, batch   100, loss 0.447
epoch     5, batch   200, loss 0.437
epoch     5, batch   300, loss 0.432
epoch     5, batch   400, loss 0.430
Accuracy of the network on the 1

## Task 7

In [9]:
class MyConvNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Input: (B, 1, 28, 28)
        self.conv_block1 = nn.Sequential(
            # (B, 1, 28, 28) -> (B, 32, 28, 28)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # (B, 32, 28, 28) -> (B, 32, 14, 14)
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Input: (B, 32, 14, 14)
        self.conv_block2 = nn.Sequential(
            # (B, 32, 14, 14) -> (B, 64, 14, 14)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # (B, 64, 14, 14) -> (B, 64, 7, 7)
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Input: (B, 64, 7, 7)
        self.fc_block = nn.Sequential(
            # (B, 64, 7, 7) -> (B, 64 * 7 * 7) = (B, 3136)
            nn.Flatten(),
            # (B, 3136) -> (B, 128)
            nn.Linear(in_features=64 * 7 * 7, out_features=128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            # (B, 128) -> (B, 10)
            nn.Linear(in_features=128, out_features=10)
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.fc_block(x)
        return x

In [10]:
epochs = 5

net_2 = MyConvNet()
trainer = MnistTrainer(batch_size=128)
trainer.train(net_2, epochs=epochs)

epoch     1, batch   100, loss 0.306
epoch     1, batch   200, loss 0.121
epoch     1, batch   300, loss 0.099
epoch     1, batch   400, loss 0.086
Accuracy of the network on the 10000 test images: 98.67%
epoch     2, batch   100, loss 0.067
epoch     2, batch   200, loss 0.066
epoch     2, batch   300, loss 0.062
epoch     2, batch   400, loss 0.061
Accuracy of the network on the 10000 test images: 98.45%
epoch     3, batch   100, loss 0.048
epoch     3, batch   200, loss 0.051
epoch     3, batch   300, loss 0.046
epoch     3, batch   400, loss 0.041
Accuracy of the network on the 10000 test images: 99.11%
epoch     4, batch   100, loss 0.042
epoch     4, batch   200, loss 0.037
epoch     4, batch   300, loss 0.039
epoch     4, batch   400, loss 0.042
Accuracy of the network on the 10000 test images: 99.12%
epoch     5, batch   100, loss 0.028
epoch     5, batch   200, loss 0.033
epoch     5, batch   300, loss 0.031
epoch     5, batch   400, loss 0.034
Accuracy of the network on the 1